# requires-grad-leaf-assert — ex2: categorize bad optimizer params (non-leaf vs no-grad)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `requires-grad-leaf-assert`. Running the final beacon cell reports progress against the `Generative: requires_grad leaf assert` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: requires_grad leaf assert` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`requires-grad-leaf-assert`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "requires-grad-leaf-assert"
DD_SUBTOPIC = "Generative: requires_grad leaf assert"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `requires_grad` + leaf assertion — deepening refresher

An optimizer updates LEAF tensors with `requires_grad=True`. A leaf tensor has no `.grad_fn` — it's either an `nn.Parameter` or a tensor you explicitly constructed with `requires_grad=True`. Op outputs are non-leaves; `opt.step()` silently skips them.

**The classic silent bug.** You call `.to('cpu')` (or `.to(device)`) on an `nn.Parameter` to move it. The return value is a NEW non-leaf tensor — `.to()` on a tensor that requires grad treats the move as an op, so the result has a `grad_fn`. You hand the result to `Adam([...], lr=...)`, the optimizer accepts it, runs `.step()` without errors, and updates NOTHING. Training loss plateaus, you debug the model for a day. The fix: move the whole `nn.Module` (`model.to(device)`) — Module's `.to()` rebinds parameters in place.

**The diagnostic pattern.** Given a list of would-be optimizer params, audit each one and return a STRUCTURED report — which params are leaf (safe), which are not (silent skip), which have `requires_grad=False` (also silent skip).

### Exercise 2 — categorize bad optimizer params (non-leaf vs no-grad)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze a list of would-be optimizer params and produce a structured report of which indices are non-leaf, which lack requires_grad, and which are safe to optimize.
> Keywords: leaf, requires_grad, diagnosis, optimizer-debug
> ```

**KCs targeted:** `categorize-leaf-vs-non-leaf`, `report-all-failing-params`

Implement `ex2_audit_optim_params(params)`. Unlike the ex1 assertion helper which raises on the FIRST bad param, this deepening exercise produces a full report so the user can see every problem at once.

For each tensor `p` at index `i`:

- If `p.is_leaf is True` AND `p.requires_grad is True` ⇒ the param is SAFE for the optimizer. Skip it.
- Else, append a dict to a `problems` list:
  ```python
  {
      'index': i,
      'shape': tuple(p.shape),
      'is_leaf': bool(p.is_leaf),
      'requires_grad': bool(p.requires_grad),
      'category': <'non-leaf' | 'no-grad'>,
  }
  ```
- The `category` field is `'non-leaf'` whenever `p.is_leaf is False` (regardless of `requires_grad`), and `'no-grad'` when the param IS a leaf but has `requires_grad=False`. Non-leaf takes priority because a non-leaf tensor is the more fundamental break: even if you fixed `requires_grad`, the optimizer still couldn't update it.

Return the `problems` list. If every param is safe, return `[]`. Do NOT raise.

Input: list of `Tensor` (possibly `nn.Parameter`).
Output: `list[dict]` — empty iff every param is optimizer-safe.

In [ ]:
def ex2_audit_optim_params(params):
    problems = []
    for i, p in enumerate(params):
        bad_leaf = not p.is_leaf
        bad_grad = not p.requires_grad
        if not bad_leaf and not bad_grad:
            continue
        # Non-leaf is the more fundamental break — report it first.
        category = 'non-leaf' if bad_leaf else 'no-grad'
        problems.append({
            'index': i,
            'shape': tuple(p.shape),
            'is_leaf': bool(p.is_leaf),
            'requires_grad': bool(p.requires_grad),
            'category': category,
        })
    return problems


<details><summary>Solution</summary>

```python
def ex2_audit_optim_params(params):
    problems = []
    for i, p in enumerate(params):
        bad_leaf = not p.is_leaf
        bad_grad = not p.requires_grad
        if not bad_leaf and not bad_grad:
            continue
        # Non-leaf is the more fundamental break — report it first.
        category = 'non-leaf' if bad_leaf else 'no-grad'
        problems.append({
            'index': i,
            'shape': tuple(p.shape),
            'is_leaf': bool(p.is_leaf),
            'requires_grad': bool(p.requires_grad),
            'category': category,
        })
    return problems
```

**Why a report instead of an assertion.** The ex1 assertion is correct for a TIGHT defensive check at optimizer construction — fail fast on the first problem. This deepening drill is for the DEBUGGING workflow: you've already hit a silent-no-op bug and want to see every offender in one pass. Categorizing them tells you which fix to apply: non-leaf usually means 'you `.to(device)`'d a Parameter — move the Module instead'; no-grad usually means 'you froze this Parameter intentionally — exclude it from the optimizer list'.

**Why only two categories, not three.** PyTorch enforces that any tensor with `requires_grad=False` is automatically a leaf (by docs: 'tensors that have requires_grad=False will be leaf Tensors by convention'). So a 'non-leaf + no-grad' specimen can't be constructed via normal means — operating on a Parameter inside `no_grad` yields a LEAF tensor with `requires_grad=False`. Reporting non-leaf takes priority because it's the more fundamental break — even if you fixed `requires_grad`, the optimizer still couldn't update a non-leaf.

**Why non-leaf priority.** When both flags are bad (rare in practice, but possible with custom autograd ops), saying 'non-leaf' first directs the user to the structural fix ('move the whole Module, not the parameter'), which usually restores `requires_grad=True` as a side effect.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()